# Same MLP, Keras Edition

Identical architecture as the PyTorch notebook, written in Keras. Note how `model.fit` replaces the explicit training loop.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
tf.keras.utils.set_random_seed(SEED)

df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet')
df = pd.get_dummies(df, columns=['device_type', 'country'], drop_first=True)
for c in ['email_risk', 'device_entropy']:
    df[c] = df[c].fillna(df[c].median())
y = df['is_fraud'].values.astype(np.float32)
X = df.drop(columns=['is_fraud']).values.astype(np.float32)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_tr)
X_tr, X_va = scaler.transform(X_tr).astype(np.float32), scaler.transform(X_va).astype(np.float32)

In [ ]:
def build_model(in_dim, hidden=(64, 32), dropout=0.2):
    inputs = keras.Input(shape=(in_dim,))
    x = inputs
    for h in hidden:
        x = keras.layers.Dense(h)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(1)(x)   # logits
    return keras.Model(inputs, outputs)

model = build_model(X_tr.shape[1])

# Class imbalance via class_weight or pos_weight-equivalent loss.
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
class_weight = {0: 1.0, 1: float(neg / pos)}

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.AUC(name='roc_auc'),
             keras.metrics.AUC(name='pr_auc', curve='PR')],
)
model.summary()

In [ ]:
model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=10, batch_size=256,
    class_weight=class_weight,
    verbose=2,
)

logits = model.predict(X_va, verbose=0).ravel()
probs = 1 / (1 + np.exp(-logits))
print(f"\nROC-AUC: {roc_auc_score(y_va, probs):.4f}")
print(f"PR-AUC : {average_precision_score(y_va, probs):.4f}")

### Side-by-side mental model

| PyTorch | Keras |
|---|---|
| `nn.Linear(in, out)` | `Dense(out)` (infers `in`) |
| `model.train()` / `model.eval()` | Handled automatically by `fit`/`evaluate` |
| Manual epoch loop | `model.fit(...)` |
| `BCEWithLogitsLoss(pos_weight=...)` | `BinaryCrossentropy(from_logits=True)` + `class_weight` |
| `torch.utils.data.DataLoader` | `tf.data.Dataset` |

Same math, different ergonomics.